In [ ]:

import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from optimization.generators import generate_problem_data
from optimization.losses import make_quadratic_loss
from optimization.runner import run_optimization
from optimization.optimizers.collinear import CollinearGradients


# конфигурация

N_VALUES = [10, 50, 100, 200, 500, 1000, 2000, 3000, 4000, 5000]
KAPPA_VALUES = [10, 50, 100]
NUM_U0 = 3 # три начальных приближения

NUM_ITERS = 3000
TOL_DIST = 0.05

LR = 1e-2
BETAS_ADAM = (0.9, 0.999)
MOMENTUM_NESTEROV = 0.9

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

MAX_ITERS_MAP = {
    "Adam": NUM_ITERS,
    "Метод Нестерова": NUM_ITERS,
    "Метод коллинеарных градиентов": 50,
}

OPTIMIZER_NAMES = [
    "Adam",
    "Метод Нестерова",
    "Метод коллинеарных градиентов",
]


# сиды

def seeds_for(n: int, kappa: int) -> tuple[int, int, list[int]]:
    """Формирует seed для A, b и трёх начальных приближений."""
    base = 1000 * n + 100 * kappa
    return base, base + 1, [base + 2, base + 3, base + 4]


# оптимизаторы

def make_optimizers(loss_fn):
    return {
        "Adam": (torch.optim.Adam, dict(lr=LR, betas=BETAS_ADAM)),
        # Nesterov = SGD с momentum и nesterov=True
        "Метод Нестерова": (torch.optim.SGD, dict(lr=LR,
                                                 momentum=MOMENTUM_NESTEROV,
                                                 nesterov=True)),
        "Метод коллинеарных градиентов": (CollinearGradients, dict(
            lr=1.0,
            objective_fn=loss_fn,
            c1=1e-8,
            c2=4.0,
            delta0=1e-5,
            h0=1e-8,
        )),
    }


# эксперимент

def run_experiment():
    results = {
        kappa: {
            u0_idx: {n: {} for n in N_VALUES}
            for u0_idx in range(NUM_U0)
        }
        for kappa in KAPPA_VALUES
    }

    for kappa in KAPPA_VALUES:
        print(f"\n==================== κ = {kappa} ====================")
        for n in N_VALUES:
            seed_A, seed_b, u0_seeds = seeds_for(n, kappa)

            # генерируем A, b один раз — они общие для всех u0 и методов
            data_base = generate_problem_data(
                n=n, kappa=float(kappa),
                seed_A=seed_A, seed_b=seed_b, seed_u0=u0_seeds[0],
                R=5.0,
            )
            loss_fn = make_quadratic_loss(data_base["A"], data_base["b"])
            optimizers = make_optimizers(loss_fn)

            for u0_idx, seed_u0 in enumerate(u0_seeds):
                # пересобираем задачу с конкретным seed_u0,
                data = generate_problem_data(
                    n=n, kappa=float(kappa),
                    seed_A=seed_A, seed_b=seed_b, seed_u0=seed_u0,
                    R=5.0,
                )

                for name, (opt_class, opt_kwargs) in optimizers.items():
                    u_final, hist = run_optimization(
                        opt_class,
                        u0_np=data["u0"],
                        loss_fn=loss_fn,
                        u_star_np=data["u_star"],
                        num_iters=MAX_ITERS_MAP[name],
                        tol_dist=TOL_DIST,
                        **opt_kwargs,
                    )
                    results[kappa][u0_idx][n][name] = {
                        "time": hist["time"],
                        "total_evals": hist["f_evals"] + hist["grad_evals"],
                    }

            # короткий лог
            parts = []
            for u0_idx in range(NUM_U0):
                for name in optimizers:
                    r = results[kappa][u0_idx][n][name]
                    parts.append(f"u0#{u0_idx+1}/{name[:3]}: {r['time']:.3f}s")
            print(f"  n={n:<5} " + "  ".join(parts[:3]))   # лог только по u0#1

    return results


# запуск

t0 = time.perf_counter()
results = run_experiment()
print(f"\nВсе эксперименты заняли {time.perf_counter() - t0:.1f} с")


==================== κ = 10 ====================
  n=10    u0#1/Ada: 0.173s  u0#1/Мет: 0.010s  u0#1/Мет: 0.003s
  n=50    u0#1/Ada: 0.130s  u0#1/Мет: 0.008s  u0#1/Мет: 0.003s
  n=100   u0#1/Ada: 0.063s  u0#1/Мет: 0.009s  u0#1/Мет: 0.003s
  n=200   u0#1/Ada: 0.050s  u0#1/Мет: 0.008s  u0#1/Мет: 0.003s
  n=500   u0#1/Ada: 0.035s  u0#1/Мет: 0.010s  u0#1/Мет: 0.004s
  n=1000  u0#1/Ada: 0.041s  u0#1/Мет: 0.016s  u0#1/Мет: 0.006s
  n=2000  u0#1/Ada: 0.091s  u0#1/Мет: 0.058s  u0#1/Мет: 0.022s
  n=3000  u0#1/Ada: 0.257s  u0#1/Мет: 0.173s  u0#1/Мет: 0.062s
  n=4000  u0#1/Ada: 0.450s  u0#1/Мет: 0.307s  u0#1/Мет: 0.114s
  n=5000  u0#1/Ada: 0.719s  u0#1/Мет: 0.485s  u0#1/Мет: 0.179s

==================== κ = 50 ====================
  n=10    u0#1/Ada: 0.355s  u0#1/Мет: 0.004s  u0#1/Мет: 0.003s
  n=50    u0#1/Ada: 0.200s  u0#1/Мет: 0.005s  u0#1/Мет: 0.013s
  n=100   u0#1/Ada: 0.103s  u0#1/Мет: 0.005s  u0#1/Мет: 0.004s
  n=200   u0#1/Ada: 0.111s  u0#1/Мет: 0.005s  u0#1/Мет: 0.017s
  n=500   u0#1/Ada

In [ ]:
# графики

COLORS = {
    "Adam": "C1",
    "Метод Нестерова": "C2",
    "Метод коллинеарных градиентов": "C0",
}
MARKERS = {
    "Adam": "s",
    "Метод Нестерова": "^",
    "Метод коллинеарных градиентов": "o",
}

# Обозначения графиков
PLOT_LABELS = ["а)", "б)", "в)", "г)", "д)", "е)"]

for kappa in KAPPA_VALUES:
    fig, axes = plt.subplots(3, 2, figsize=(14, 12), facecolor="white")
    fig.suptitle(f"κ = {kappa}", fontsize=16, fontweight="bold", color="black")

    for u0_idx in range(NUM_U0):
        ax_time = axes[u0_idx, 0]
        ax_evals = axes[u0_idx, 1]

        for name in OPTIMIZER_NAMES:
            times = [results[kappa][u0_idx][n][name]["time"] for n in N_VALUES]
            evals = [results[kappa][u0_idx][n][name]["total_evals"] for n in N_VALUES]

            ax_time.plot(
                N_VALUES, times,
                marker=MARKERS[name],
                color=COLORS[name],
                label=name
            )
            ax_evals.plot(
                N_VALUES, evals,
                marker=MARKERS[name],
                color=COLORS[name],
                label=name
            )

        for ax in (ax_time, ax_evals):
            ax.set_facecolor("white")
            ax.tick_params(colors="black")
            ax.xaxis.label.set_color("black")
            ax.yaxis.label.set_color("black")
            ax.title.set_color("black")
            ax.ticklabel_format(style="plain", axis="both", useOffset=False)

            for spine in ax.spines.values():
                spine.set_color("black")

        # левый столбец: время
        plot_idx = u0_idx * 2

        ax_time.set_xlabel("Размерность n")
        ax_time.set_ylabel("Время, с")
        ax_time.set_title(
            f"{PLOT_LABELS[plot_idx]} Время вычислений при $u_0^{{({u0_idx + 1})}}$"
        )
        ax_time.grid(True, alpha=0.3)
        ax_time.legend(
            fontsize=9,
            facecolor="white",
            edgecolor="black",
            labelcolor="black"
        )

        # правый столбец: количество вычислений
        ax_evals.set_xlabel("Размерность n")
        ax_evals.set_ylabel(r"Вычислений $f + \nabla f$, шт.")
        ax_evals.set_title(
            f"{PLOT_LABELS[plot_idx + 1]} Количество вычислений при $u_0^{{({u0_idx + 1})}}$"
        )
        ax_evals.grid(True, alpha=0.3)
        ax_evals.legend(
            fontsize=9,
            facecolor="white",
            edgecolor="black",
            labelcolor="black"
        )

    fig.tight_layout(rect=(0, 0, 1, 0.97))

    out_path = RESULTS_DIR / f"comparison_kappa{kappa}.png"
    fig.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="white"
    )

    plt.show()
    print(f"Сохранено: {out_path.resolve()}")